# El Niño Teleconnection Precipitation Patterns Analysis
## Investigation of EP vs CP El Niños in CESM piControl and 6ka Simulations

This notebook analyzes the teleconnection patterns of Eastern Pacific (EP) and Central Pacific (CP) El Niños by:
1. Loading SST and precipitation data from CESM simulations
2. Computing SST anomalies
3. Calculating E and C indices using PCA
4. Creating composite precipitation maps based on these indices
5. **NEW: Focusing on DJF (December-January-February) seasonal precipitation**

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy import stats
import xESMF as xe
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

## 1. Data Loading and Preprocessing

In [ ]:
# Define paths to your CESM data
# Modify these paths according to your data directory structure
sst_pi_path = 'path/to/piControl_SST_data.nc'  # Update with your path
sst_6ka_path = 'path/to/6ka_SST_data.nc'        # Update with your path
precip_pi_path = 'path/to/piControl_PRECIP_data.nc'  # Update with your path
precip_6ka_path = 'path/to/6ka_PRECIP_data.nc'      # Update with your path

# Load SST and precipitation data
print("Loading data...")
sst_pi = xr.open_dataset(sst_pi_path)
sst_6ka = xr.open_dataset(sst_6ka_path)
precip_pi = xr.open_dataset(precip_pi_path)
precip_6ka = xr.open_dataset(precip_6ka_path)

print(f"piControl SST shape: {sst_pi.dims}")
print(f"6ka SST shape: {sst_6ka.dims}")
print(f"piControl Precip shape: {precip_pi.dims}")
print(f"6ka Precip shape: {precip_6ka.dims}")

## 2. Regridding Function

In [ ]:
def regrid(ds, target_ds=None):
    """
    Regrid CESM data to 1x1 degree global grid using bilinear interpolation.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset with TLONG and TLAT coordinates
    target_ds : xarray.Dataset, optional
        Target grid definition. If None, uses ds's lat/lon
    
    Returns:
    --------
    xarray.Dataset
        Regridded dataset on 1x1 degree grid
    """
    dr = ds.copy()
    
    # 1. Rename the coordinate names to lon and lat
    # because xESMF has no way to guess variable meaning
    if "TLONG" in ds.coords and "TLAT" in ds.coords:
        dr = dr.rename({"TLONG": "lon", "TLAT": "lat"})
    elif "lon" not in ds.coords or "lat" not in ds.coords:
        print("Warning: Dataset doesn't have expected coordinates")
        print(f"Available coordinates: {list(ds.coords)}")
    
    # 2. Create output grid (1 degree x 1 degree global grid)
    if target_ds is None:
        ds_out = xr.Dataset(
            {
                "lat": (["lat"], np.arange(-90, 90.1, 1.0), {"units": "degrees_north"}),
                "lon": (["lon"], np.arange(-180, 180.0, 1.0), {"units": "degrees_east"}),
            }
        )
    else:
        ds_out = target_ds
    
    # 3. Perform regridding
    print(f"Regridding from {dr.sizes} to {ds_out.sizes}...")
    try:
        regridder = xe.Regridder(dr, ds_out, "bilinear")
        dr_out = regridder(dr)
        print("Regridding complete!")
        return dr_out
    except Exception as e:
        print(f"Error during regridding: {e}")
        return dr

In [ ]:
# Apply regridding to all datasets
print("Regridding SST and Precipitation data...\n")

# Create common target grid
target_grid = xr.Dataset(
    {
        "lat": (["lat"], np.arange(-90, 90.1, 1.0), {"units": "degrees_north"}),
        "lon": (["lon"], np.arange(-180, 180.0, 1.0), {"units": "degrees_east"}),
    }
)

# Regrid all datasets
print("Regridding piControl SST...")
sst_pi_regrid = regrid(sst_pi, target_grid)

print("\nRegridding 6ka SST...")
sst_6ka_regrid = regrid(sst_6ka, target_grid)

print("\nRegridding piControl Precipitation...")
precip_pi_regrid = regrid(precip_pi, target_grid)

print("\nRegridding 6ka Precipitation...")
precip_6ka_regrid = regrid(precip_6ka, target_grid)

print("\nAll regridding complete!")

## 3. Seasonal Filtering Helper Functions

In [ ]:
def get_month_from_time(time_coord):
    """
    Extract month from time coordinate.
    Handles various time formats.
    """
    try:
        # Convert to pandas datetime if needed
        times = pd.to_datetime(time_coord.values)
        return times.month.values
    except:
        # Fallback: assume numeric month-like values
        return np.array([int(str(t).split('-')[1]) if '-' in str(t) else np.nan for t in time_coord.values])

def select_season(ds, months, time_dim='time'):
    """
    Select specific months from dataset (e.g., DJF for months 12, 1, 2).
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset with time dimension
    months : list or tuple
        Months to select (e.g., [12, 1, 2] for DJF)
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    xarray.Dataset
        Dataset with selected months only
    """
    month_indices = get_month_from_time(ds[time_dim])
    mask = np.isin(month_indices, months)
    
    return ds.isel({time_dim: mask})

def compute_seasonal_climatology(ds, months, time_dim='time'):
    """
    Compute seasonal climatology by averaging over selected months.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset
    months : list or tuple
        Months for the season
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    xarray.Dataset
        Seasonal climatology
    """
    seasonal_data = select_season(ds, months, time_dim)
    return seasonal_data.mean(dim=time_dim)

# Define seasonal month groups
SEASONAL_MONTHS = {
    'DJF': (12, 1, 2),  # December-January-February (Boreal Winter)
    'MAM': (3, 4, 5),   # March-April-May (Boreal Spring)
    'JJA': (6, 7, 8),   # June-July-August (Boreal Summer)
    'SON': (9, 10, 11)  # September-October-November (Boreal Fall)
}

print("Seasonal filtering functions loaded!")

## 4. Compute SST Anomalies

In [ ]:
def compute_anomalies(ds, time_dim='time'):
    """
    Compute anomalies by removing climatological mean.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Input dataset
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    xarray.Dataset
        Dataset with anomalies
    """
    # Calculate climatological mean
    climatology = ds.mean(dim=time_dim)
    
    # Compute anomalies
    anomalies = ds - climatology
    
    return anomalies, climatology

# Compute anomalies for SST
print("Computing SST anomalies...\n")
sst_pi_anom, sst_pi_clim = compute_anomalies(sst_pi_regrid)
sst_6ka_anom, sst_6ka_clim = compute_anomalies(sst_6ka_regrid)

print(f"piControl SST anomalies shape: {sst_pi_anom.dims}")
print(f"6ka SST anomalies shape: {sst_6ka_anom.dims}")

# Display summary statistics
print(f"\npiControl SST anomalies - min: {sst_pi_anom.min().values:.2f}, max: {sst_pi_anom.max().values:.2f}")
print(f"6ka SST anomalies - min: {sst_6ka_anom.min().values:.2f}, max: {sst_6ka_anom.max().values:.2f}")

## 5. Calculate E and C Indices Using PCA

In [ ]:
def calculate_nino_indices(sst_anom, lon_range_e=[150, 360], lon_range_c=[120, 150], 
                          lat_range=[-5, 5], time_dim='time'):
    """
    Calculate EP and CP El Niño indices from SST anomalies using PCA.
    
    Parameters:
    -----------
    sst_anom : xarray.Dataset
        SST anomalies
    lon_range_e : list
        Longitude range for EP Nino region [W, E] in degrees
    lon_range_c : list
        Longitude range for CP Nino region [W, E] in degrees
    lat_range : list
        Latitude range for both regions [S, N] in degrees
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    dict
        Dictionary containing E and C indices, PCA objects, and regional SST
    """
    
    # Extract SST data variable (handle different possible names)
    sst_var = None
    for var in ['SST', 'TS', 'TEMP', 'temperature']:
        if var in sst_anom.data_vars:
            sst_var = sst_anom[var]
            break
    
    if sst_var is None:
        # Use first data variable
        sst_var = list(sst_anom.data_vars.values())[0]
        print(f"Using variable: {sst_var.name}")
    
    # Select equatorial region and time series
    sst_eq = sst_var.sel(lat=slice(lat_range[0], lat_range[1]))
    
    # Extract EP and CP regions
    sst_ep = sst_eq.sel(lon=slice(lon_range_e[0], lon_range_e[1]))
    sst_cp = sst_eq.sel(lon=slice(lon_range_c[0], lon_range_c[1]))
    
    print(f"EP region shape: {sst_ep.shape}")
    print(f"CP region shape: {sst_cp.shape}")
    
    # Reshape for PCA
    # Convert to 2D: (time, space)
    time_len = sst_ep.sizes['time']
    
    # EP region
    sst_ep_2d = sst_ep.values.reshape(time_len, -1)
    # Remove NaN values
    mask_ep = ~np.isnan(sst_ep_2d).any(axis=0)
    sst_ep_2d = sst_ep_2d[:, mask_ep]
    
    # CP region
    sst_cp_2d = sst_cp.values.reshape(time_len, -1)
    mask_cp = ~np.isnan(sst_cp_2d).any(axis=0)
    sst_cp_2d = sst_cp_2d[:, mask_cp]
    
    print(f"EP region valid points: {sst_ep_2d.shape[1]}")
    print(f"CP region valid points: {sst_cp_2d.shape[1]}")
    
    # Standardize the data
    sst_ep_std = (sst_ep_2d - sst_ep_2d.mean(axis=0)) / (sst_ep_2d.std(axis=0) + 1e-10)
    sst_cp_std = (sst_cp_2d - sst_cp_2d.mean(axis=0)) / (sst_cp_2d.std(axis=0) + 1e-10)
    
    # Apply PCA
    pca_ep = PCA(n_components=3)
    pca_cp = PCA(n_components=3)
    
    pc_ep = pca_ep.fit_transform(sst_ep_std)
    pc_cp = pca_cp.fit_transform(sst_cp_std)
    
    # Get E and C indices from first principal components
    E_index = pc_ep[:, 0]  # EP index
    C_index = pc_cp[:, 0]  # CP index
    
    # Standardize indices
    E_index = (E_index - E_index.mean()) / E_index.std()
    C_index = (C_index - C_index.mean()) / C_index.std()
    
    print(f"\nVariance explained by first PC:")
    print(f"  EP: {pca_ep.explained_variance_ratio_[0]:.2%}")
    print(f"  CP: {pca_cp.explained_variance_ratio_[0]:.2%}")
    
    results = {
        'E_index': E_index,
        'C_index': C_index,
        'pca_ep': pca_ep,
        'pca_cp': pca_cp,
        'sst_ep': sst_ep,
        'sst_cp': sst_cp,
        'pc_ep': pc_ep,
        'pc_cp': pc_cp
    }
    
    return results

print("Calculating E and C indices for piControl...\n")
indices_pi = calculate_nino_indices(sst_pi_anom)

print("\n" + "="*60)
print("Calculating E and C indices for 6ka...\n")
indices_6ka = calculate_nino_indices(sst_6ka_anom)

## 6. Create Composite Precipitation Maps - ALL SEASONS

In [ ]:
def create_precipitation_composites(precip_data, E_index, C_index, 
                                   threshold_ep=0.5, threshold_cp=0.5,
                                   time_dim='time'):
    """
    Create composite precipitation maps based on E and C indices.
    
    Parameters:
    -----------
    precip_data : xarray.Dataset
        Precipitation data
    E_index : np.array
        EP El Niño index
    C_index : np.array
        CP El Niño index
    threshold_ep : float
        Threshold for EP events (in standard deviations)
    threshold_cp : float
        Threshold for CP events (in standard deviations)
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    dict
        Dictionary containing composite maps and event masks
    """
    
    # Extract precipitation variable
    precip_var = None
    for var in ['PRECIP', 'PRECT', 'precip', 'precipitation']:
        if var in precip_data.data_vars:
            precip_var = precip_data[var]
            break
    
    if precip_var is None:
        # Use first data variable
        precip_var = list(precip_data.data_vars.values())[0]
        print(f"Using precipitation variable: {precip_var.name}")
    
    # Create masks for EP and CP events
    ep_positive = E_index > threshold_ep
    ep_negative = E_index < -threshold_ep
    cp_positive = C_index > threshold_cp
    cp_negative = C_index < -threshold_cp
    
    print(f"EP positive events: {ep_positive.sum()}")
    print(f"EP negative events: {ep_negative.sum()}")
    print(f"CP positive events: {cp_positive.sum()}")
    print(f"CP negative events: {cp_negative.sum()}")
    
    # Create composites
    composite_ep_pos = precip_var.isel({time_dim: ep_positive}).mean(dim=time_dim)
    composite_ep_neg = precip_var.isel({time_dim: ep_negative}).mean(dim=time_dim)
    composite_ep_anom = composite_ep_pos - composite_ep_neg
    
    composite_cp_pos = precip_var.isel({time_dim: cp_positive}).mean(dim=time_dim)
    composite_cp_neg = precip_var.isel({time_dim: cp_negative}).mean(dim=time_dim)
    composite_cp_anom = composite_cp_pos - composite_cp_neg
    
    results = {
        'composite_ep_pos': composite_ep_pos,
        'composite_ep_neg': composite_ep_neg,
        'composite_ep_anom': composite_ep_anom,
        'composite_cp_pos': composite_cp_pos,
        'composite_cp_neg': composite_cp_neg,
        'composite_cp_anom': composite_cp_anom,
        'ep_positive_mask': ep_positive,
        'cp_positive_mask': cp_positive,
        'n_ep_pos': ep_positive.sum(),
        'n_ep_neg': ep_negative.sum(),
        'n_cp_pos': cp_positive.sum(),
        'n_cp_neg': cp_negative.sum()
    }
    
    return results

print("Creating precipitation composites for piControl (all seasons)...\n")
composites_pi = create_precipitation_composites(
    precip_pi_regrid, 
    indices_pi['E_index'], 
    indices_pi['C_index']
)

print("\n" + "="*60)
print("Creating precipitation composites for 6ka (all seasons)...\n")
composites_6ka = create_precipitation_composites(
    precip_6ka_regrid, 
    indices_6ka['E_index'], 
    indices_6ka['C_index']
)

## 7. NEW: DJF-Specific Analysis

In [ ]:
# Extract DJF data for SST
print("Extracting DJF (December-January-February) data...\n")
sst_pi_djf = select_season(sst_pi_regrid, SEASONAL_MONTHS['DJF'])
sst_6ka_djf = select_season(sst_6ka_regrid, SEASONAL_MONTHS['DJF'])

precip_pi_djf = select_season(precip_pi_regrid, SEASONAL_MONTHS['DJF'])
precip_6ka_djf = select_season(precip_6ka_regrid, SEASONAL_MONTHS['DJF'])

print(f"piControl DJF SST shape: {sst_pi_djf.dims}")
print(f"6ka DJF SST shape: {sst_6ka_djf.dims}")
print(f"piControl DJF Precip shape: {precip_pi_djf.dims}")
print(f"6ka DJF Precip shape: {precip_6ka_djf.dims}")

# Compute DJF anomalies
print("\nComputing DJF SST anomalies...")
sst_pi_djf_anom, sst_pi_djf_clim = compute_anomalies(sst_pi_djf)
sst_6ka_djf_anom, sst_6ka_djf_clim = compute_anomalies(sst_6ka_djf)

# Calculate DJF E and C indices
print("\nCalculating DJF E and C indices for piControl...")
indices_pi_djf = calculate_nino_indices(sst_pi_djf_anom)

print("\n" + "="*60)
print("Calculating DJF E and C indices for 6ka...")
indices_6ka_djf = calculate_nino_indices(sst_6ka_djf_anom)

In [ ]:
# Create DJF precipitation composites
print("Creating DJF precipitation composites for piControl...\n")
composites_pi_djf = create_precipitation_composites(
    precip_pi_djf,
    indices_pi_djf['E_index'],
    indices_pi_djf['C_index']
)

print("\n" + "="*60)
print("Creating DJF precipitation composites for 6ka...\n")
composites_6ka_djf = create_precipitation_composites(
    precip_6ka_djf,
    indices_6ka_djf['E_index'],
    indices_6ka_djf['C_index']
)

## 8. Visualize DJF Composite Precipitation Maps

In [ ]:
def plot_composites(composites, title_prefix, vmin=-5, vmax=5, cmap='RdBu_r'):
    """
    Plot composite precipitation maps.
    
    Parameters:
    -----------
    composites : dict
        Dictionary containing composite maps
    title_prefix : str
        Prefix for plot titles
    vmin, vmax : float
        Colorbar limits
    cmap : str
        Colormap name
    """
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12), 
                              subplot_kw=dict(projection=None))
    
    # EP anomaly
    im1 = axes[0, 0].contourf(composites['composite_ep_anom'].lon, 
                              composites['composite_ep_anom'].lat,
                              composites['composite_ep_anom'].values,
                              levels=20, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, 0].set_title(f'{title_prefix}: EP El Niño Precipitation Anomaly', 
                         fontsize=12, fontweight='bold')
    axes[0, 0].set_xlabel('Longitude')
    axes[0, 0].set_ylabel('Latitude')
    cbar1 = plt.colorbar(im1, ax=axes[0, 0], label='Precip Anomaly (mm/day)')
    axes[0, 0].grid(True, alpha=0.3)
    
    # CP anomaly
    im2 = axes[0, 1].contourf(composites['composite_cp_anom'].lon,
                              composites['composite_cp_anom'].lat,
                              composites['composite_cp_anom'].values,
                              levels=20, cmap=cmap, vmin=vmin, vmax=vmax)
    axes[0, 1].set_title(f'{title_prefix}: CP El Niño Precipitation Anomaly',
                         fontsize=12, fontweight='bold')
    axes[0, 1].set_xlabel('Longitude')
    axes[0, 1].set_ylabel('Latitude')
    cbar2 = plt.colorbar(im2, ax=axes[0, 1], label='Precip Anomaly (mm/day)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # EP positive composite
    im3 = axes[1, 0].contourf(composites['composite_ep_pos'].lon,
                              composites['composite_ep_pos'].lat,
                              composites['composite_ep_pos'].values,
                              levels=20, cmap='Blues')
    axes[1, 0].set_title(f'{title_prefix}: EP Positive Phase Precipitation (n={composites["n_ep_pos"]})',
                         fontsize=12, fontweight='bold')
    axes[1, 0].set_xlabel('Longitude')
    axes[1, 0].set_ylabel('Latitude')
    cbar3 = plt.colorbar(im3, ax=axes[1, 0], label='Precip (mm/day)')
    axes[1, 0].grid(True, alpha=0.3)
    
    # CP positive composite
    im4 = axes[1, 1].contourf(composites['composite_cp_pos'].lon,
                              composites['composite_cp_pos'].lat,
                              composites['composite_cp_pos'].values,
                              levels=20, cmap='Blues')
    axes[1, 1].set_title(f'{title_prefix}: CP Positive Phase Precipitation (n={composites["n_cp_pos"]})',
                         fontsize=12, fontweight='bold')
    axes[1, 1].set_xlabel('Longitude')
    axes[1, 1].set_ylabel('Latitude')
    cbar4 = plt.colorbar(im4, ax=axes[1, 1], label='Precip (mm/day)')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    return fig

# Plot DJF piControl composites
print("Plotting DJF composites for piControl...")
fig_pi_djf = plot_composites(composites_pi_djf, 'piControl DJF')
plt.savefig('Composites_piControl_DJF.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot DJF 6ka composites
print("Plotting DJF composites for 6ka...")
fig_6ka_djf = plot_composites(composites_6ka_djf, '6ka DJF')
plt.savefig('Composites_6ka_DJF.png', dpi=300, bbox_inches='tight')
plt.show()

print("DJF figures saved!")

## 9. DJF Comparison: piControl vs 6ka

In [ ]:
# Create DJF comparison plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# EP Anomaly comparison (DJF)
im1 = axes[0, 0].contourf(composites_pi_djf['composite_ep_anom'].lon,
                          composites_pi_djf['composite_ep_anom'].lat,
                          composites_pi_djf['composite_ep_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[0, 0].set_title('piControl DJF: EP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0, 0], label='mm/day')
axes[0, 0].grid(True, alpha=0.3)

im2 = axes[0, 1].contourf(composites_6ka_djf['composite_ep_anom'].lon,
                          composites_6ka_djf['composite_ep_anom'].lat,
                          composites_6ka_djf['composite_ep_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[0, 1].set_title('6ka DJF: EP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[0, 1], label='mm/day')
axes[0, 1].grid(True, alpha=0.3)

# CP Anomaly comparison (DJF)
im3 = axes[1, 0].contourf(composites_pi_djf['composite_cp_anom'].lon,
                          composites_pi_djf['composite_cp_anom'].lat,
                          composites_pi_djf['composite_cp_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[1, 0].set_title('piControl DJF: CP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
plt.colorbar(im3, ax=axes[1, 0], label='mm/day')
axes[1, 0].grid(True, alpha=0.3)

im4 = axes[1, 1].contourf(composites_6ka_djf['composite_cp_anom'].lon,
                          composites_6ka_djf['composite_cp_anom'].lat,
                          composites_6ka_djf['composite_cp_anom'].values,
                          levels=20, cmap='RdBu_r', vmin=-5, vmax=5)
axes[1, 1].set_title('6ka DJF: CP El Niño Precip Anomaly', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Longitude')
plt.colorbar(im4, ax=axes[1, 1], label='mm/day')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('Comparison_DJF_EP_CP_Anomalies.png', dpi=300, bbox_inches='tight')
plt.show()

print("DJF Comparison figure saved!")

## 10. Statistical Analysis and Significance (DJF)

In [ ]:
def compute_statistical_significance(precip_data, E_index, C_index, time_dim='time'):
    """
    Compute t-statistics for composite differences.
    
    Parameters:
    -----------
    precip_data : xarray.Dataset
        Precipitation data
    E_index : np.array
        EP El Niño index
    C_index : np.array
        CP El Niño index
    time_dim : str
        Name of time dimension
    
    Returns:
    --------
    dict
        Dictionary containing t-statistics and p-values
    """
    
    # Extract precipitation variable
    precip_var = None
    for var in ['PRECIP', 'PRECT', 'precip', 'precipitation']:
        if var in precip_data.data_vars:
            precip_var = precip_data[var]
            break
    
    if precip_var is None:
        precip_var = list(precip_data.data_vars.values())[0]
    
    # Create masks
    ep_positive = E_index > 0.5
    ep_negative = E_index < -0.5
    cp_positive = C_index > 0.5
    cp_negative = C_index < -0.5
    
    # Extract positive and negative phases
    ep_pos_data = precip_var.isel({time_dim: ep_positive})
    ep_neg_data = precip_var.isel({time_dim: ep_negative})
    cp_pos_data = precip_var.isel({time_dim: cp_positive})
    cp_neg_data = precip_var.isel({time_dim: cp_negative})
    
    # Compute t-statistics
    ep_mean_pos = ep_pos_data.mean(dim=time_dim)
    ep_mean_neg = ep_neg_data.mean(dim=time_dim)
    ep_std_pos = ep_pos_data.std(dim=time_dim)
    ep_std_neg = ep_neg_data.std(dim=time_dim)
    
    ep_n_pos = ep_pos_data.sizes[time_dim]
    ep_n_neg = ep_neg_data.sizes[time_dim]
    
    # Welch's t-test
    ep_se = np.sqrt(ep_std_pos**2/ep_n_pos + ep_std_neg**2/ep_n_neg)
    ep_t_stat = (ep_mean_pos - ep_mean_neg) / ep_se
    
    # Same for CP
    cp_mean_pos = cp_pos_data.mean(dim=time_dim)
    cp_mean_neg = cp_neg_data.mean(dim=time_dim)
    cp_std_pos = cp_pos_data.std(dim=time_dim)
    cp_std_neg = cp_neg_data.std(dim=time_dim)
    
    cp_n_pos = cp_pos_data.sizes[time_dim]
    cp_n_neg = cp_neg_data.sizes[time_dim]
    
    cp_se = np.sqrt(cp_std_pos**2/cp_n_pos + cp_std_neg**2/cp_n_neg)
    cp_t_stat = (cp_mean_pos - cp_mean_neg) / cp_se
    
    return {
        'ep_t_stat': ep_t_stat,
        'cp_t_stat': cp_t_stat,
        'ep_mean_diff': ep_mean_pos - ep_mean_neg,
        'cp_mean_diff': cp_mean_pos - cp_mean_neg
    }

print("Computing statistical significance for DJF...\n")
stats_pi_djf = compute_statistical_significance(precip_pi_djf, 
                                                indices_pi_djf['E_index'], 
                                                indices_pi_djf['C_index'])
stats_6ka_djf = compute_statistical_significance(precip_6ka_djf,
                                                 indices_6ka_djf['E_index'],
                                                 indices_6ka_djf['C_index'])

print("DJF Statistical significance computed!")

In [ ]:
# Plot DJF t-statistics
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# piControl EP t-stat (DJF)
im1 = axes[0, 0].contourf(stats_pi_djf['ep_t_stat'].lon,
                          stats_pi_djf['ep_t_stat'].lat,
                          stats_pi_djf['ep_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[0, 0].set_title('piControl DJF: EP El Niño t-statistic', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Latitude')
plt.colorbar(im1, ax=axes[0, 0], label='t-value')
axes[0, 0].grid(True, alpha=0.3)

# 6ka EP t-stat (DJF)
im2 = axes[0, 1].contourf(stats_6ka_djf['ep_t_stat'].lon,
                          stats_6ka_djf['ep_t_stat'].lat,
                          stats_6ka_djf['ep_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[0, 1].set_title('6ka DJF: EP El Niño t-statistic', fontsize=12, fontweight='bold')
plt.colorbar(im2, ax=axes[0, 1], label='t-value')
axes[0, 1].grid(True, alpha=0.3)

# piControl CP t-stat (DJF)
im3 = axes[1, 0].contourf(stats_pi_djf['cp_t_stat'].lon,
                          stats_pi_djf['cp_t_stat'].lat,
                          stats_pi_djf['cp_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[1, 0].set_title('piControl DJF: CP El Niño t-statistic', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Longitude')
axes[1, 0].set_ylabel('Latitude')
plt.colorbar(im3, ax=axes[1, 0], label='t-value')
axes[1, 0].grid(True, alpha=0.3)

# 6ka CP t-stat (DJF)
im4 = axes[1, 1].contourf(stats_6ka_djf['cp_t_stat'].lon,
                          stats_6ka_djf['cp_t_stat'].lat,
                          stats_6ka_djf['cp_t_stat'].values,
                          levels=20, cmap='RdBu_r', vmin=-3, vmax=3)
axes[1, 1].set_title('6ka DJF: CP El Niño t-statistic', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Longitude')
plt.colorbar(im4, ax=axes[1, 1], label='t-value')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('T_Statistics_DJF.png', dpi=300, bbox_inches='tight')
plt.show()

print("DJF T-statistics figure saved!")

## 11. Summary Statistics Comparison: DJF vs All Seasons

In [ ]:
# Create comprehensive summary table
print("\n" + "="*90)
print("SUMMARY STATISTICS: DJF vs ALL SEASONS")
print("="*90)

summary_data = {
    'Metric': [],
    'piControl All': [],
    'piControl DJF': [],
    '6ka All': [],
    '6ka DJF': []
}

metrics = [
    ('E Index Mean', indices_pi['E_index'].mean(), indices_pi_djf['E_index'].mean(),
     indices_6ka['E_index'].mean(), indices_6ka_djf['E_index'].mean()),
    ('E Index Std', indices_pi['E_index'].std(), indices_pi_djf['E_index'].std(),
     indices_6ka['E_index'].std(), indices_6ka_djf['E_index'].std()),
    ('C Index Mean', indices_pi['C_index'].mean(), indices_pi_djf['C_index'].mean(),
     indices_6ka['C_index'].mean(), indices_6ka_djf['C_index'].mean()),
    ('C Index Std', indices_pi['C_index'].std(), indices_pi_djf['C_index'].std(),
     indices_6ka['C_index'].std(), indices_6ka_djf['C_index'].std()),
    ('E-C Correlation', np.corrcoef(indices_pi['E_index'], indices_pi['C_index'])[0,1],
     np.corrcoef(indices_pi_djf['E_index'], indices_pi_djf['C_index'])[0,1],
     np.corrcoef(indices_6ka['E_index'], indices_6ka['C_index'])[0,1],
     np.corrcoef(indices_6ka_djf['E_index'], indices_6ka_djf['C_index'])[0,1]),
    ('EP Events (>0.5σ)', (indices_pi['E_index'] > 0.5).sum(), (indices_pi_djf['E_index'] > 0.5).sum(),
     (indices_6ka['E_index'] > 0.5).sum(), (indices_6ka_djf['E_index'] > 0.5).sum()),
    ('CP Events (>0.5σ)', (indices_pi['C_index'] > 0.5).sum(), (indices_pi_djf['C_index'] > 0.5).sum(),
     (indices_6ka['C_index'] > 0.5).sum(), (indices_6ka_djf['C_index'] > 0.5).sum())
]

for metric_name, val_pi_all, val_pi_djf, val_6ka_all, val_6ka_djf in metrics:
    summary_data['Metric'].append(metric_name)
    if isinstance(val_pi_all, (int, np.integer)):
        summary_data['piControl All'].append(f"{val_pi_all}")
        summary_data['piControl DJF'].append(f"{val_pi_djf}")
        summary_data['6ka All'].append(f"{val_6ka_all}")
        summary_data['6ka DJF'].append(f"{val_6ka_djf}")
    else:
        summary_data['piControl All'].append(f"{val_pi_all:.3f}")
        summary_data['piControl DJF'].append(f"{val_pi_djf:.3f}")
        summary_data['6ka All'].append(f"{val_6ka_all:.3f}")
        summary_data['6ka DJF'].append(f"{val_6ka_djf:.3f}")

summary_df = pd.DataFrame(summary_data)
print(summary_df.to_string(index=False))
print("="*90)

## 12. Save DJF Results

In [ ]:
# Save DJF indices as NetCDF for future use
print("Saving DJF results...\n")

# Create datasets for DJF indices
indices_pi_djf_ds = xr.Dataset(
    {
        'E_index': (['time'], indices_pi_djf['E_index']),
        'C_index': (['time'], indices_pi_djf['C_index'])
    },
    coords={'time': range(len(indices_pi_djf['E_index']))}
)
indices_pi_djf_ds.to_netcdf('El_Nino_Indices_piControl_DJF.nc')
print("Saved: El_Nino_Indices_piControl_DJF.nc")

indices_6ka_djf_ds = xr.Dataset(
    {
        'E_index': (['time'], indices_6ka_djf['E_index']),
        'C_index': (['time'], indices_6ka_djf['C_index'])
    },
    coords={'time': range(len(indices_6ka_djf['E_index']))}
)
indices_6ka_djf_ds.to_netcdf('El_Nino_Indices_6ka_DJF.nc')
print("Saved: El_Nino_Indices_6ka_DJF.nc")

# Save composite anomalies for DJF
composites_pi_djf['composite_ep_anom'].to_netcdf('Composite_EP_Anomaly_piControl_DJF.nc')
print("Saved: Composite_EP_Anomaly_piControl_DJF.nc")

composites_pi_djf['composite_cp_anom'].to_netcdf('Composite_CP_Anomaly_piControl_DJF.nc')
print("Saved: Composite_CP_Anomaly_piControl_DJF.nc")

composites_6ka_djf['composite_ep_anom'].to_netcdf('Composite_EP_Anomaly_6ka_DJF.nc')
print("Saved: Composite_EP_Anomaly_6ka_DJF.nc")

composites_6ka_djf['composite_cp_anom'].to_netcdf('Composite_CP_Anomaly_6ka_DJF.nc')
print("Saved: Composite_CP_Anomaly_6ka_DJF.nc")

print("\nAll DJF results saved successfully!")

## 13. Key Findings: DJF Analysis

In [ ]:
print("\n" + "="*90)
print("KEY FINDINGS: DJF SEASON ANALYSIS")
print("="*90)

print("\n1. DJF VERSUS ALL SEASONS:")
print("   piControl:")
print(f"     - All seasons E-C correlation: {np.corrcoef(indices_pi['E_index'], indices_pi['C_index'])[0,1]:.3f}")
print(f"     - DJF E-C correlation: {np.corrcoef(indices_pi_djf['E_index'], indices_pi_djf['C_index'])[0,1]:.3f}")
print("   6ka:")
print(f"     - All seasons E-C correlation: {np.corrcoef(indices_6ka['E_index'], indices_6ka['C_index'])[0,1]:.3f}")
print(f"     - DJF E-C correlation: {np.corrcoef(indices_6ka_djf['E_index'], indices_6ka_djf['C_index'])[0,1]:.3f}")

print("\n2. DJF EVENT FREQUENCY:")
print(f"   piControl DJF: EP={composites_pi_djf['n_ep_pos']} events, CP={composites_pi_djf['n_cp_pos']} events")
print(f"   6ka DJF: EP={composites_6ka_djf['n_ep_pos']} events, CP={composites_6ka_djf['n_cp_pos']} events")

print("\n3. DJF PRECIPITATION PATTERNS:")
print("   - DJF is the boreal winter season, peak El Niño mature phase")
print("   - Maximum teleconnection signals expected in this season")
print("   - Strong precipitation anomalies expected over Pacific and global regions")
print("   - Compare EP vs CP responses for distinct teleconnection patterns")

print("\n4. DIFFERENCES BETWEEN SIMULATIONS (DJF):")
print("   - Check if orbital forcing (6ka vs piControl) affects DJF ENSO impacts")
print("   - Evaluate changes in teleconnection strength during peak season")
print("   - Assess Holocene climate sensitivity to orbital parameters")

print("\n" + "="*90)